# ASL Recognition Complete Notebook

## 1. Setup & Installation

In [ ]:
# Install dependencies jika diperlukan
# !pip install tensorflow mediapipe opencv-python scikit-learn matplotlib seaborn tqdm gtts

In [ ]:
# Import libraries
import os
import cv2
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import mediapipe as mp

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.20.0
GPU available: []


## 2. Constants & Configuration

In [ ]:
# ASL Class Names (29 classes)
CLASS_NAMES = [
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J',
    'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T',
    'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space'
]
NUM_CLASSES = len(CLASS_NAMES)
INPUT_DIM = 126  # 2 Tangan x 21 Titik x 3 Koordinat (X,Y,Z)

# Sesuaikan dengan path dataset lu ngab
TRAIN_DIR = r"E:\Project\dataset\bisindo\images\train"
MODEL_FAST_PATH = 'saved_models/mobilenet_landmark.keras'
MODEL_ACC_PATH = 'saved_models/efficientnet_landmark.keras'

print(f"Jumlah Kelas: {NUM_CLASSES}")
print(f"Dimensi Input: {INPUT_DIM}")

Number of classes: 29
Image size: (200, 200)


## 3. Data Loading Module

In [ ]:
def load_asl_dataset(data_dir, img_size=DEFAULT_IMG_SIZE, normalize=True, verbose=True):
    """Load ASL alphabet dataset from directory."""
    images = []
    labels = []
    available_classes = [c for c in CLASS_NAMES if os.path.exists(os.path.join(data_dir, c))]
    
    if verbose:
        print(f"Loading dataset from: {data_dir}")
        print(f"Found {len(available_classes)} classes")
    
    for class_idx, class_name in enumerate(tqdm(available_classes) if verbose else available_classes):
        class_dir = os.path.join(data_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        for img_name in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_name)
            try:
                img = Image.open(img_path).convert('RGB')
                img = img.resize(img_size, Image.Resampling.LANCZOS)
                images.append(np.array(img))
                labels.append(class_idx)
            except Exception as e:
                continue
    
    images = np.array(images, dtype=np.float32)
    labels = np.array(labels, dtype=np.int32)
    if normalize:
        images = images / 255.0
    if verbose:
        print(f"Loaded {len(images)} images")
    return images, labels, available_classes


def create_data_generators(train_dir, img_size=DEFAULT_IMG_SIZE, batch_size=32, validation_split=0.2):
    """Create data generators with augmentation."""
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.1,
        brightness_range=[0.8, 1.2],
        fill_mode='nearest',
        validation_split=validation_split
    )
    
    train_gen = train_datagen.flow_from_directory(
        train_dir, target_size=img_size, batch_size=batch_size,
        class_mode='categorical', subset='training', shuffle=True, seed=42
    )
    val_gen = train_datagen.flow_from_directory(
        train_dir, target_size=img_size, batch_size=batch_size,
        class_mode='categorical', subset='validation', shuffle=False, seed=42
    )
    return train_gen, val_gen, train_gen.class_indices

## 4. Model Architecture

In [4]:
def create_mobilenet_model(num_classes=NUM_CLASSES, input_shape=(*DEFAULT_IMG_SIZE, 3), dropout_rate=0.5):
    """Create ASL classifier using MobileNetV2 as base model."""
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False
    
    inputs = Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate/2)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def create_landmark_model(num_classes=NUM_CLASSES, input_dim=63):
    """Create landmark-based classifier model."""
    inputs = Input(shape=(input_dim,))
    x = Dense(256, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu')(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


# Preview model
model = create_mobilenet_model()
model.summary()

/var/folders/hw/jxcjts5x3tq7n14008rpjl2c0000gn/T/ipykernel_36509/2517752808.py:3: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 200, 200, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 29)             │         7,453 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,599,517 (9.92 MB)

 Trainable params: 338,461 (1.29 MB)

 Non-trainable params: 2,261,056 (8.63 MB)

## 5. Training CNN Model

In [6]:
# Training Configuration
EPOCHS = 30
BATCH_SIZE = 32

if os.path.exists(TRAIN_DIR):
    print("Dataset found! Starting training...")
    
    train_gen, val_gen, class_indices = create_data_generators(TRAIN_DIR, batch_size=BATCH_SIZE)
    print(f"Training samples: {train_gen.samples}")
    print(f"Validation samples: {val_gen.samples}")
    
    # Auto-detect classes dari dataset
    num_classes_detected = len(class_indices)
    print(f"Classes detected: {num_classes_detected}")
    
    # Buat model dengan jumlah class yang benar
    model = create_mobilenet_model(num_classes=num_classes_detected)
    
    os.makedirs('saved_models', exist_ok=True)
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_accuracy', save_best_only=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7)
    ]
    
    history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks)
    print(f"Model saved to: {MODEL_SAVE_PATH}")
else:
    print(f"Dataset not found at: {TRAIN_DIR}")

Dataset found! Starting training...
The history saving thread hit an unexpected error (OperationalError('unable to open database file')).History will not be written to the database.
Found 140426 images belonging to 23 classes.
Found 35092 images belonging to 23 classes.
Training samples: 140426
Validation samples: 35092
Classes detected: 23


/var/folders/hw/jxcjts5x3tq7n14008rpjl2c0000gn/T/ipykernel_36509/2517752808.py:3: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape)


Epoch 1/30
  19/4389 ━━━━━━━━━━━━━━━━━━━━ 20:25 280ms/step - accuracy: 0.0755 - loss: 3.8883

KeyboardInterrupt: 

## 6. Hand Detection (MediaPipe)

In [10]:
from dataclasses import dataclass
from typing import List, Tuple
import os

@dataclass
class HandResult:
    landmarks: np.ndarray
    bbox: Tuple[int, int, int, int]
    handedness: str
    confidence: float


class HandDetector:
    """Hand detection using MediaPipe (compatible with 0.10.x)."""
    
    def __init__(self, max_num_hands=2, min_detection_confidence=0.7):
        # Try legacy API first, then new API
        try:
            self.mp_hands = mp.solutions.hands
            self.mp_drawing = mp.solutions.drawing_utils
            self.mp_styles = mp.solutions.drawing_styles
            self.hands = self.mp_hands.Hands(
                static_image_mode=False, max_num_hands=max_num_hands,
                min_detection_confidence=min_detection_confidence
            )
            self.use_legacy = True
        except AttributeError:
            # MediaPipe 0.10.x+ uses Tasks API
            from mediapipe.tasks import python
            from mediapipe.tasks.python import vision
            
            model_path = 'hand_landmarker.task'
            if not os.path.exists(model_path):
                import urllib.request
                url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
                print("Downloading hand landmarker model...")
                urllib.request.urlretrieve(url, model_path)
            
            base_options = python.BaseOptions(model_asset_path=model_path)
            options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=max_num_hands)
            self.detector = vision.HandLandmarker.create_from_options(options)
            self.use_legacy = False
    
    def detect(self, frame, draw=False):
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        hands = []
        h, w = frame.shape[:2]
        
        if self.use_legacy:
            results = self.hands.process(rgb)
            if results.multi_hand_landmarks:
                for idx, hand_lm in enumerate(results.multi_hand_landmarks):
                    lm = np.array([[l.x, l.y, l.z] for l in hand_lm.landmark], dtype=np.float32)
                    x_coords = (lm[:, 0] * w).astype(int)
                    y_coords = (lm[:, 1] * h).astype(int)
                    pad = 40
                    x, y = max(0, x_coords.min()-pad), max(0, y_coords.min()-pad)
                    x2, y2 = min(w, x_coords.max()+pad), min(h, y_coords.max()+pad)
                    handedness = results.multi_handedness[idx].classification[0].label if results.multi_handedness else 'Right'
                    conf = results.multi_handedness[idx].classification[0].score if results.multi_handedness else 0.0
                    hands.append(HandResult(lm, (x, y, x2-x, y2-y), handedness, conf))
        else:
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            results = self.detector.detect(mp_image)
            if results.hand_landmarks:
                for idx, hand_lm in enumerate(results.hand_landmarks):
                    lm = np.array([[l.x, l.y, l.z] for l in hand_lm], dtype=np.float32)
                    x_coords = (lm[:, 0] * w).astype(int)
                    y_coords = (lm[:, 1] * h).astype(int)
                    pad = 40
                    x, y = max(0, x_coords.min()-pad), max(0, y_coords.min()-pad)
                    x2, y2 = min(w, x_coords.max()+pad), min(h, y_coords.max()+pad)
                    handedness = results.handedness[idx][0].category_name if results.handedness else 'Right'
                    conf = results.handedness[idx][0].score if results.handedness else 0.0
                    hands.append(HandResult(lm, (x, y, x2-x, y2-y), handedness, conf))
        return hands
    
    def release(self):
        if self.use_legacy and hasattr(self, 'hands'):
            self.hands.close()

print("HandDetector class defined (compatible with MediaPipe 0.10.x)!")

HandDetector class defined (compatible with MediaPipe 0.10.x)!


## 7. Extract Landmarks from Dataset

In [11]:
def extract_landmarks_from_dataset(data_dir, max_per_class=100):
    """Extract hand landmarks from dataset images."""
    detector = HandDetector(max_num_hands=1)
    all_landmarks = []
    all_labels = []
    
    available_classes = [c for c in CLASS_NAMES if os.path.exists(os.path.join(data_dir, c))]
    
    for class_idx, class_name in enumerate(tqdm(available_classes)):
        class_dir = os.path.join(data_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        
        count = 0
        for img_name in os.listdir(class_dir):
            if count >= max_per_class:
                break
            img_path = os.path.join(class_dir, img_name)
            try:
                img = cv2.imread(img_path)
                if img is None:
                    continue
                hands = detector.detect(img, draw=False)
                if hands:
                    lm = hands[0].landmarks
                    # Normalize
                    wrist = lm[0].copy()
                    normalized = lm - wrist
                    max_dist = np.max(np.linalg.norm(normalized, axis=1))
                    if max_dist > 0:
                        normalized = normalized / max_dist
                    all_landmarks.append(normalized.flatten())
                    all_labels.append(class_idx)
                    count += 1
            except:
                continue
    
    detector.release()
    return np.array(all_landmarks), np.array(all_labels)

# Run extraction if dataset exists
if os.path.exists(TRAIN_DIR):
    print("Extracting landmarks...")
    X_landmarks, y_landmarks = extract_landmarks_from_dataset(TRAIN_DIR, max_per_class=100)
    print(f"Extracted {len(X_landmarks)} landmark samples")
    np.savez('saved_models/landmarks_train.npz', X=X_landmarks, y=y_landmarks)
else:
    print("Dataset not found. Skip landmark extraction.")

Extracting landmarks...


OSError: [Errno 28] No space left on device

## 8. Train Landmark Classifier

In [ ]:
# Load or use extracted landmarks
if os.path.exists('saved_models/landmarks_train.npz'):
    data = np.load('saved_models/landmarks_train.npz')
    X = data['X']
    y = data['y']
    print(f"Loaded {len(X)} landmark samples")
    
    # Split data
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    y_train_cat = to_categorical(y_train, NUM_CLASSES)
    y_val_cat = to_categorical(y_val, NUM_CLASSES)
    
    # Create and train model
    landmark_model = create_landmark_model()
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ModelCheckpoint(LANDMARK_MODEL_PATH, monitor='val_accuracy', save_best_only=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
    ]
    
    history_lm = landmark_model.fit(
        X_train, y_train_cat,
        validation_data=(X_val, y_val_cat),
        epochs=50,
        batch_size=32,
        callbacks=callbacks
    )
    
    # Save class names
    np.save('saved_models/landmark_classifier_classes.npy', CLASS_NAMES)
    print(f"Landmark model saved to: {LANDMARK_MODEL_PATH}")
else:
    print("Landmarks not found. Run extraction first.")

## 9. Evaluation & Visualization

In [ ]:
def plot_training_history(history, title='Training History'):
    """Plot training history."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Val')
    axes[0].set_title(f'{title} - Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True)
    
    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Val')
    axes[1].set_title(f'{title} - Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150)
    plt.show()


def plot_confusion_matrix(y_true, y_pred, classes):
    """Plot confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(15, 12))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=150)
    plt.show()


# Plot history if available
if 'history' in dir():
    plot_training_history(history, 'CNN Model')
if 'history_lm' in dir():
    plot_training_history(history_lm, 'Landmark Model')

In [ ]:
# Evaluate landmark model
if os.path.exists(LANDMARK_MODEL_PATH) and 'X_val' in dir():
    model_eval = tf.keras.models.load_model(LANDMARK_MODEL_PATH)
    predictions = model_eval.predict(X_val)
    y_pred = np.argmax(predictions, axis=1)
    
    # Classification report
    unique_classes = np.unique(y_val)
    class_labels = [CLASS_NAMES[i] for i in unique_classes]
    print(classification_report(y_val, y_pred, target_names=class_labels))
    
    # Confusion matrix
    plot_confusion_matrix(y_val, y_pred, class_labels)
    
    accuracy = np.sum(y_pred == y_val) / len(y_val)
    print(f"\nOverall Accuracy: {accuracy:.4f}")

## 10. Test Inference (Real-time Demo)

In [ ]:
class LandmarkClassifier:
    """Landmark-based ASL classifier."""
    
    def __init__(self, model_path=LANDMARK_MODEL_PATH):
        self.model = None
        self.class_names = CLASS_NAMES
        if os.path.exists(model_path):
            self.model = tf.keras.models.load_model(model_path)
            print(f"Model loaded: {model_path}")
    
    def normalize(self, landmarks):
        if landmarks is None:
            return None
        normalized = landmarks - landmarks[0]
        max_dist = np.max(np.linalg.norm(normalized, axis=1))
        if max_dist > 0:
            normalized = normalized / max_dist
        return normalized
    
    def predict(self, landmarks):
        if self.model is None or landmarks is None:
            return '?', 0.0
        normalized = self.normalize(landmarks)
        flat = normalized.flatten().reshape(1, -1)
        pred = self.model.predict(flat, verbose=0)[0]
        idx = np.argmax(pred)
        return self.class_names[idx], float(pred[idx])

# Test
if os.path.exists(LANDMARK_MODEL_PATH):
    classifier = LandmarkClassifier()
    print("Classifier ready for inference!")

In [ ]:
# Real-time demo (uncomment to run)
# WARNING: This will open your webcam

# def run_demo():
#     detector = HandDetector(max_num_hands=1)
#     classifier = LandmarkClassifier()
#     cap = cv2.VideoCapture(0)
#     
#     while cap.isOpened():
#         ret, frame = cap.read()
#         if not ret:
#             break
#         frame = cv2.flip(frame, 1)
#         hands = detector.detect(frame, draw=True)
#         
#         if hands:
#             letter, conf = classifier.predict(hands[0].landmarks)
#             cv2.putText(frame, f'{letter}: {conf:.0%}', (10, 50),
#                        cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
#         
#         cv2.imshow('ASL Recognition', frame)
#         if cv2.waitKey(1) & 0xFF == ord('q'):
#             break
#     
#     cap.release()
#     cv2.destroyAllWindows()
#     detector.release()

# run_demo()

---
## Summary

Notebook ini berisi:
1. ✅ Data loading & preprocessing
2. ✅ CNN model architecture (MobileNetV2)
3. ✅ Training pipeline
4. ✅ Hand detection (MediaPipe)
5. ✅ Landmark extraction & training
6. ✅ Evaluation & visualization
7. ✅ Real-time inference demo

**Untuk menjalankan training:**
1. Tambahkan dataset ke `dataset/asl_alphabet_train/`
2. Jalankan semua cells dari atas ke bawah